In [3]:
import pandas as pd

# 1. Create a mock transaction history for a user
data = {
    "user_id": [101, 101, 101, 101, 101],
    "timestamp": pd.to_datetime([
        "2026-05-18 10:00:00",
        "2026-05-18 10:02:00", # 2 mins from tx 1
        "2026-05-18 10:04:00", # 4 mins from tx 1 (3rd tx in 5m)
        "2026-05-18 10:20:00", # Quiet period
        "2026-05-18 10:21:00"  # Only 2 tx in this window
    ]),
    "amount": [50, 100, 30, 200, 15]
}
df = pd.DataFrame(data).sort_values(["user_id", "timestamp"])
df


,user_id,timestamp,amount
0,101,2026-05-18 10:00:00,50
1,101,2026-05-18 10:02:00,100
2,101,2026-05-18 10:04:00,30
3,101,2026-05-18 10:20:00,200
4,101,2026-05-18 10:21:00,15


In [4]:
# 2. Calculate rolling transaction count in a 5-minute window per user
df["tx_count_5m"] = (
    df.groupby("user_id")
    .rolling("5min", on="timestamp")["timestamp"]
    .count()
    .values
)

# 3. Flag as suspicious if there are 3 or more transactions
df["velocity_fraud_flag"] = df["tx_count_5m"] >= 3

# Display results
print(df[["timestamp", "amount", "tx_count_5m", "velocity_fraud_flag"]])


            timestamp  amount  tx_count_5m  velocity_fraud_flag
0 2026-05-18 10:00:00      50          1.0                False
1 2026-05-18 10:02:00     100          2.0                False
2 2026-05-18 10:04:00      30          3.0                 True
3 2026-05-18 10:20:00     200          1.0                False
4 2026-05-18 10:21:00      15          2.0                False
